In [5]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import (
    GemmaReplacement,
    SAEReplacementLayer,
    make_replacement_model,
)

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 1
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 1
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 1

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model.eval()
    model.requires_grad_(False)
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [2]:
# import torch
# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer

# from transformers_sae import _autoreload
# from transformers_sae.ops import MemoryTrackingMode
# from transformers_sae.replacement_model import (
#     make_replacement_model,
# )

# # Tweak TRAINING_BATCH_SIZE for your hardware if necessary
# if torch.cuda.is_available():
#     TRAINING_DEVICE = "cuda:0"
#     TRAINING_BATCH_SIZE = 16
# elif torch.mps.is_available():
#     TRAINING_DEVICE = "mps:0"
#     TRAINING_BATCH_SIZE = 8
# else:
#     TRAINING_DEVICE = "cpu"
#     TRAINING_BATCH_SIZE = 8

# tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
# training_dataset = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
# validation_dataset = load_dataset(
#     "roneneldan/TinyStories", split="validation", streaming=True
# )

# with MemoryTrackingMode() as mtm:
#     model = AutoModelForCausalLM.from_pretrained(
#         "roneneldan/TinyStories-33M",
#         dtype=torch.bfloat16,
#         device_map=TRAINING_DEVICE,
#         use_safetensors=True,
#     )
#     model.requires_grad_(False)
#     model = make_replacement_model(
#         model,
#         {},
#         num_layers=model.config.num_hidden_layers,
#         context_length=512,
#         d_model=model.config.hidden_size,
#     )

# print(model)
# print(mtm.memory_max, mtm.memory_cur)

In [3]:
from transformers_sae.ops import find_latest_checkpoint, load_checkpoint

checkpoint_dir = ".checkpoints/next_layer/"

checkpoint = find_latest_checkpoint(checkpoint_dir, 0)
sae = load_checkpoint(checkpoint).sae
sae.config.device = torch.device(TRAINING_DEVICE)
sae.encoder.config.device = torch.device(TRAINING_DEVICE)
sae.decoder.config.device = torch.device(TRAINING_DEVICE)

In [9]:
import numpy as np

from transformers_sae.sae import SAE, make_sae_config
from transformers_sae.validation import run_validations


class NoisySAE(SAE):
    def __init__(self, *args, **kwargs):
        self.layer = kwargs.pop("layer")
        super().__init__(*args, **kwargs)
        self.encoder = torch.nn.Identity()

    def forward(self, *args, **kwargs):
        original_dtype = args[0].dtype
        original_output = args[0].float()
        if self.layer >= 0:
            random_direction = torch.randn_like(original_output)
            random_direction = random_direction / (
                torch.linalg.vector_norm(random_direction, dim=-1, keepdim=True) + 1e-8
            )
            original_norm = torch.linalg.vector_norm(
                original_output, dim=-1, keepdim=True
            )
            reconstruction = original_output + random_direction * original_norm * 0.50
        else:
            reconstruction = original_output
        self.encoder(
            torch.ones(
                (args[0].shape[0], args[0].shape[1], 1),
                device=TRAINING_DEVICE,
            )
        )
        return reconstruction.to(original_dtype)


sae_config = make_sae_config(
    d_model=1,
    d_sae=1,
    device=model.device,
    train_dtype=torch.float32,
    inference_dtype=torch.bfloat16,
    encoder_kind="relu",
)

TRAINING_BATCH_SIZE = 1
TOKENIZER_BATCH_SIZE = 256
NUM_TOKENS = int(1e4)

validations = run_validations(
    model,
    tokenizer,
    {
        layer: NoisySAE(sae_config, layer=layer) # if layer > 0 else sae
        for layer in range(model.num_layers)
    },
    training_dataset,
    TOKENIZER_BATCH_SIZE,
    TRAINING_BATCH_SIZE,
    NUM_TOKENS,
)

print(
    f"mean rre={ {k: np.mean(v.rre).item() for k, v in validations.layer_results.items() if v.rre is not None} }"
)
print(
    f"geom mean kl={ {k: np.exp(np.mean(np.log(np.clip(v.kl, min=1e-9)))).item() for k, v in validations.layer_results.items() if v.kl is not None} }"
)

Running SAE evals:   0%|          | 0/10000 [00:00<?, ?it/s]

mean rre={0: 0.5000004768371582, 1: 0.6919741034507751, 2: 0.9482443332672119, 3: 1.082039475440979, 4: 1.1688092947006226, 5: 1.236767053604126, 6: 1.3526041507720947, 7: 1.360437273979187, 8: 1.4819433689117432, 9: 1.5031806230545044, 10: 1.4634909629821777, 11: 1.4809010028839111, 12: 1.6254574060440063, 13: 1.528563141822815, 14: 1.4730249643325806, 15: 1.5453341007232666, 16: 1.55465567111969, 17: 1.5499050617218018, 18: 1.612348198890686, 19: 1.6074708700180054, 20: 1.5968685150146484, 21: 1.6068775653839111, 22: 1.7203562259674072, 23: 1.8806909322738647, 24: 1.8986414670944214, 25: 1.9027312994003296}
geom mean kl={26: 11.099000930786133}
